In [2]:
# import optuna
# from optuna.samplers import TPESampler
import warnings

import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import math
from collections import defaultdict
import seaborn as sns
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Lasso, Ridge, RidgeCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier
from pathlib import Path

warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # mount your Drive

# Then refer to your file like this:
NOTES_CSV = "/content/drive/MyDrive/MIMIC_IV_Note/note/discharge.csv"


## Simple LLM-based extraction pipeline

In [ ]:
# Example: LLM-based phenotype extraction from clinical notes
import pandas as pd
from tqdm import tqdm

# Option 1: using a pretrained clinical / medical NER model
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Option 2: using a more generic LLM & prompt classification
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

################ CONFIG ################
NOTES_CSV = r"E:/Chrome Dls/MIMIC_IV_Note/note/discharge.csv"
OUTPUT_PHENO_CSV = "../data/processed/note_pheno_extracted.csv"
TEXT_COL = "text"
SUBJ_COL = "subject_id"
BATCH_SIZE = 64   # adjust to memory & speed
MAX_LEN = 1024    # may lower if memory issues
########################################

### Choose model for NER or classification
# Example NER model — suppose a medical-NER checkpoint; replace with actual model name
NER_MODEL_NAME = "d4data/biomedical-ner-all"  # example; choose licensed & suitable model
tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0)  # device=-1 for CPU

def extract_phenotypes_ner(text):
    """Run NER pipeline and return list of unique phenotype/entity strings."""
    try:
        ents = ner(text[:MAX_LEN])
        # Example: return only disease/disorder entities — filter by ent['entity_group'] or label
        phenos = [e['word'] for e in ents if e.get('entity_group') in ("DISEASE","DISORDER","CONDITION","ANATOMY","SYMPTOM")]
        phenos = list(set(phenos))
        return phenos
    except Exception as e:
        return []

### (Alternative) Option 2: prompt-based extraction using seq2seq LLM
LLM_MODEL = "google/flan-t5-base"  # or another open-source LLM
tok2 = AutoTokenizer.from_pretrained(LLM_MODEL)
lm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

def extract_phenotypes_prompt(text, phenos_of_interest=None):
    """
    Simple prompt: ask LLM to list phenotypes / diagnoses mentioned in note.
    phenos_of_interest : optional list of strings to check for.
    """
    prompt = "Extract all diseases, conditions or relevant diagnoses mentioned in the following clinical note. " \
             "Return a JSON array of diagnosis names. Clinical note:\n" + text[:MAX_LEN]
    inputs = tok2(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    outputs = lm.generate(**inputs, max_new_tokens=128)
    out = tok2.decode(outputs[0], skip_special_tokens=True)
    # try parse JSON from output (if LM returns JSON)
    try:
        import json
        preds = json.loads(out)
        if isinstance(preds, list):
            return preds
    except Exception:
        # fallback: simple string split
        return [p.strip() for p in out.split(",") if p.strip()]
    return []

### Main loop: read notes, extract phenotypes, aggregate per subject
df = pd.read_csv(NOTES_CSV, usecols=[SUBJ_COL, TEXT_COL], dtype={SUBJ_COL:int, TEXT_COL:str})

# iterate, maybe in chunks if large
records = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    subj = row[SUBJ_COL]
    txt = str(row[TEXT_COL])
    pheno_list = extract_phenotypes_ner(txt)  # or extract_phenotypes_prompt(txt)
    if not pheno_list:
        continue
    for p in pheno_list:
        records.append({"subject_id": subj, "phenotype": p})

pheno_df = pd.DataFrame(records)
# optionally pivot to wide: one-hot per phenotype
pheno_onehot = (pheno_df.assign(val=1)
                       .pivot_table(index="subject_id", columns="phenotype", values="val", fill_value=0)
                       .reset_index())

pheno_onehot.to_csv(OUTPUT_PHENO_CSV, index=False)
print("Saved phenotype-features to:", OUTPUT_PHENO_CSV)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'E:/Chrome Dls/MIMIC_IV_Note/note/discharge.csv'

In [38]:
# chunked_discharge_htn_extraction.py
import pandas as pd
import re
from collections import defaultdict
from datetime import datetime
import os

# -------------------- CONFIG --------------------
INPUT_CSV = r"E:/Chrome Dls/MIMIC_IV_Note/note/discharge.csv"   # your large file
OUTPUT_CSV = "../data/processed/discharge_htn_features.csv"
CHUNKSIZE = 20000   # adjust based on memory; lower if you run out of RAM
TEXT_COL = "text"
SUBJ_COL = "subject_id"
CHARTTIME_COL = "charttime"   # prefer charttime if available, else storetime
STORETIME_COL = "storetime"
# keywords to match (regex patterns). Add synonyms/abbrev as needed
KEYWORD_PATTERNS = [
    r"\bhypertension\b",
    r"\bhtn\b",
    r"\bhigh blood pressure\b",
    r"\belevated bp\b",
    r"\bhypertensive\b"
]
# negation tokens (simple set). We look for these BEFORE a keyword match (within WINDOW chars)
NEGATION_PATTERN = re.compile(
    r"\b(no|denies|denied|without|not|negative for|rule out|ruled out|absence of|free of)\b",
    flags=re.IGNORECASE
)
NEGATION_WINDOW = 120  # number of chars before keyword to check for negation words
# optionally provide anchor dates per subject to compute recency (pandas Series or DataFrame)
# anchor_df = pd.read_csv(...)  # must have columns ['subject_id', 'anchor_date'] if used
ANCHOR_DF = None  # set to your anchor DataFrame if available
# -------------------------------------------------

# compile keyword regexes
kw_regexes = [re.compile(pat, flags=re.IGNORECASE) for pat in KEYWORD_PATTERNS]

def parse_datetime_maybe(val):
    """Safely parse datetime-like string to Timestamp or return NaT."""
    if pd.isna(val):
        return pd.NaT
    try:
        return pd.to_datetime(val)
    except Exception:
        # fallback: try common formats
        try:
            return pd.to_datetime(val, errors='coerce')
        except:
            return pd.NaT

def is_negated_around(text, match_start, window=NEGATION_WINDOW):
    """
    Heuristic: look `window` chars before match_start for negation tokens.
    Returns True if a negation token is found in that window.
    """
    if not isinstance(text, str) or text == "":
        return False
    start = max(0, match_start - window)
    window_text = text[start:match_start]
    return bool(NEGATION_PATTERN.search(window_text))

# container for aggregated stats per subject
stats = defaultdict(lambda: {
    "htn_any_mention": False,
    "htn_mention_count": 0,
    "htn_first_mention_time": pd.NaT,
    "htn_last_mention_time": pd.NaT
})

# helper to update subject stats
def update_stats(subj, mention_time):
    st = stats[subj]
    st["htn_any_mention"] = True
    st["htn_mention_count"] += 1
    if pd.isna(st["htn_first_mention_time"]) or (not pd.isna(mention_time) and mention_time < st["htn_first_mention_time"]):
        st["htn_first_mention_time"] = mention_time
    if pd.isna(st["htn_last_mention_time"]) or (not pd.isna(mention_time) and mention_time > st["htn_last_mention_time"]):
        st["htn_last_mention_time"] = mention_time

# Read the CSV in chunks and process
total_rows = 0
chunks = pd.read_csv(INPUT_CSV, chunksize=CHUNKSIZE, usecols=[SUBJ_COL, CHARTTIME_COL, STORETIME_COL, TEXT_COL],
                     dtype={SUBJ_COL: int}, iterator=True)

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1} ...")
    # ensure text column strings
    chunk[TEXT_COL] = chunk[TEXT_COL].astype(str).fillna("")
    # parse times (lazily)
    chunk[CHARTTIME_COL] = pd.to_datetime(chunk[CHARTTIME_COL], errors='coerce')
    chunk[STORETIME_COL] = pd.to_datetime(chunk[STORETIME_COL], errors='coerce')

    # iterate rows (vectorizing is hard due to regex-with-window per-match)
    for idx, row in chunk.iterrows():
        total_rows += 1
        subj = int(row[SUBJ_COL])
        text = row[TEXT_COL]
        # choose mention_time: prefer charttime, else storetime, else NaT
        mention_time = row[CHARTTIME_COL] if not pd.isna(row[CHARTTIME_COL]) else row[STORETIME_COL]

        # find ANY keyword occurrences in text
        found_positive_in_row = False
        for kw_re in kw_regexes:
            for m in kw_re.finditer(text):
                start = m.start()
                # check negation window before the match
                if is_negated_around(text, start):
                    # negated mention — skip it
                    continue
                # positive mention found (non-negated)
                update_stats(subj, mention_time)
                found_positive_in_row = True
                # if you want to count multiple mentions per row, don't break; otherwise break
                # break  # optionally break to count only one mention per row
            # optionally stop checking other keywords if we already found a positive
            # if found_positive_in_row:
            #     break

    print(f"  processed rows so far: {total_rows}, unique subjects seen: {len(stats)}")

# Convert stats dict to DataFrame
rows = []
for subj, s in stats.items():
    rows.append({
        "subject_id": subj,
        "htn_any_mention": bool(s["htn_any_mention"]),
        "htn_mention_count": int(s["htn_mention_count"]),
        "htn_first_mention_time": s["htn_first_mention_time"],
        "htn_last_mention_time": s["htn_last_mention_time"]
    })

htn_df = pd.DataFrame(rows)
# ensure time columns are datetime
htn_df['htn_first_mention_time'] = pd.to_datetime(htn_df['htn_first_mention_time'])
htn_df['htn_last_mention_time'] = pd.to_datetime(htn_df['htn_last_mention_time'])

# optional: compute recency relative to anchor date if you have anchor_df (subject_id, anchor_date)
if ANCHOR_DF is not None:
    anchor_df = ANCHOR_DF.copy()
    anchor_df['anchor_date'] = pd.to_datetime(anchor_df['anchor_date'])
    htn_df = htn_df.merge(anchor_df[['subject_id','anchor_date']], on='subject_id', how='left')
    # recency in days: anchor_date - last_mention_time
    htn_df['htn_recency_days'] = (htn_df['anchor_date'] - htn_df['htn_last_mention_time']).dt.days

# Save to CSV
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
htn_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved HTN mention features to: {OUTPUT_CSV}")
print("Summary:\n", htn_df[['htn_any_mention','htn_mention_count']].describe())


Processing chunk 1 ...
  processed rows so far: 20000, unique subjects seen: 5358
Processing chunk 2 ...
  processed rows so far: 40000, unique subjects seen: 10625
Processing chunk 3 ...
  processed rows so far: 60000, unique subjects seen: 15883
Processing chunk 4 ...
  processed rows so far: 80000, unique subjects seen: 21030
Processing chunk 5 ...
  processed rows so far: 100000, unique subjects seen: 26154
Processing chunk 6 ...
  processed rows so far: 120000, unique subjects seen: 31296
Processing chunk 7 ...
  processed rows so far: 140000, unique subjects seen: 36473
Processing chunk 8 ...
  processed rows so far: 160000, unique subjects seen: 41713
Processing chunk 9 ...
  processed rows so far: 180000, unique subjects seen: 46913
Processing chunk 10 ...
  processed rows so far: 200000, unique subjects seen: 52201
Processing chunk 11 ...
  processed rows so far: 220000, unique subjects seen: 57566
Processing chunk 12 ...
  processed rows so far: 240000, unique subjects seen: 

In [39]:
htn_df.head()

,subject_id,htn_any_mention,htn_mention_count,htn_first_mention_time,htn_last_mention_time
0,10000032,True,3,2180-05-07,2180-05-07
1,10000117,True,2,2181-11-15,2183-09-21
2,10000764,True,3,2132-10-19,2132-10-19
3,10000826,True,2,2146-12-12,2147-01-02
4,10000980,True,52,2188-01-05,2193-08-17


## Optional: inspect nrows of huge csv file

In [9]:
discharge_sample = pd.read_csv("E:/Chrome Dls/MIMIC_IV_Note/note/discharge.csv", nrows=100)
discharge_sample.to_csv(
    "../data/processed/discharge_sample.csv",
    index=False,
    quoting=1,        # csv.QUOTE_ALL
    escapechar='\\'   # escape problematic characters
)

discharge_sample.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07 00:00:00,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25 00:00:00,2160-11-25 15:09:00,\nName: ___ Unit No: __...


In [37]:
print(discharge_sample.columns)
discharge_sample.head()

Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text'],
      dtype='object')


,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07 00:00:00,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25 00:00:00,2160-11-25 15:09:00,\nName: ___ Unit No: __...


## Optional: compare discharge features against Labels

In [2]:
disch_df = pd.read_csv("../data/processed/discharge_htn_features.csv")
patients_df = pd.read_csv("../data/processed/patients.csv")

In [4]:
print(len(np.unique(disch_df['subject_id'])))
print(len(np.unique(patients_df['subject_id'])))

86814
36899


In [6]:
# merge label + note features
merged = pd.merge(
    patients_df[['subject_id', 'label']],
    disch_df[['subject_id', 'htn_any_mention']],
    on='subject_id',
    how='left'
)

# total hypertensive patients
total_htn = merged[merged['label'] == 1].shape[0]

# hypertensive patients whose notes mention hypertension
htn_with_mention = merged[(merged['label'] == 1) & (merged['htn_any_mention'] == True)].shape[0]

# calculate ratio
print(f"Patients with label=1: {total_htn}")
print(f"Patients with label=1 AND mention=True: {htn_with_mention}")
print(f"Overlap ratio: {htn_with_mention / total_htn * 100:.2f}%")


Patients with label=1: 18485
Patients with label=1 AND mention=True: 12570
Overlap ratio: 68.00%


If the overlap ratio is 100%, it means every hypertensive patient has a mention in the notes → ⚠️ potential leakage (your NLP feature may be duplicating your label).

If it’s much less (e.g., 30–60%), it’s fine — this means the note feature adds real-world complementary info.

In [7]:
neg_with_mention = merged[(merged['label'] == 0) & (merged['htn_any_mention'] == True)].shape[0]
print(f"Label=0 patients with hypertension mention: {neg_with_mention}")


Label=0 patients with hypertension mention: 1822


## Optional: 
- loads discharge_htn_features.csv, admissions.csv, and your final_hosp_dataset.csv (to get labels),

- computes, per subject_id, the earliest admission time and the latest discharge time,

- compares htn_first_mention_time / htn_last_mention_time to these admission times,

- produces summary statistics showing how many note mentions occur after the associated hospitalization (possible leakage),

- saves a CSV with per-subject flags you can inspect.

In [10]:
# leakage_timing_check.py
import pandas as pd
import numpy as np
import os

# ---------------- CONFIG ----------------
DISCH_FEAT = "../data/processed/discharge_htn_features.csv"   # has subject_id, htn_first_mention_time, htn_last_mention_time
ADMISSIONS_CSV = "E:/Chrome Dls/MIMIC_IV_Core/hosp/admissions.csv"               # standard MIMIC admissions table (hadm-level)
FINAL_HOSP = "../data/processed/final_hosp_dataset.csv"      # your structured dataset with labels (subject_id, label)
OUT_SUMMARY = "../data/processed/notes_timing_leakage_summary.csv"
# ----------------------------------------

# 1) Load files
print("Loading files...")
disch = pd.read_csv(DISCH_FEAT, parse_dates=['htn_first_mention_time', 'htn_last_mention_time'], low_memory=False)
admissions = pd.read_csv(ADMISSIONS_CSV, usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime'], parse_dates=['admittime','dischtime'], low_memory=False)
final_hosp = pd.read_csv(FINAL_HOSP, usecols=['subject_id','label'], low_memory=False)  # label column name assumed 'label'

print(f"disch rows: {len(disch)}, admissions rows: {len(admissions)}, final_hosp rows: {len(final_hosp)}")

# 2) Compute per-subject admission windows: earliest admit, latest discharge
adm_agg = admissions.groupby('subject_id').agg(
    earliest_admit = ('admittime', 'min'),
    latest_discharge = ('dischtime', 'max'),
    n_admissions = ('hadm_id', 'nunique')
).reset_index()

print("Aggregated admissions (per subject) sample:")
print(adm_agg.head())

# 3) Merge discharge features with admissions aggregation and labels
merged = pd.merge(final_hosp, disch, on='subject_id', how='left')   # keep only your cohort subjects
merged = pd.merge(merged, adm_agg, on='subject_id', how='left')      # may be NaT for patients without admission rows

# 4) Ensure datetime dtypes
for col in ['htn_first_mention_time','htn_last_mention_time','earliest_admit','latest_discharge']:
    if col in merged.columns:
        merged[col] = pd.to_datetime(merged[col], errors='coerce')

# 5) Create temporal flags (per subject)
# - mention_exists: whether any mention was detected (non-null first or last)
merged['mention_exists'] = (~merged['htn_first_mention_time'].isna()) | (~merged['htn_last_mention_time'].isna())

# Use last_mention if present, else first_mention for timing checks
merged['mention_time_used'] = merged['htn_last_mention_time'].fillna(merged['htn_first_mention_time'])

# Flags:
# - mention_before_any_admit: mention_time_used <= earliest_admit  (meaning mention is before first admission in admissions table)
# - mention_during_admission_window: earliest_admit <= mention_time_used <= latest_discharge
# - mention_after_latest_discharge: mention_time_used > latest_discharge
# - no_admission_info: earliest_admit is NaT (no matching admissions in admissions.csv)
merged['no_admission_info'] = merged['earliest_admit'].isna()
merged['mention_before_any_admit'] = np.where(merged['mention_time_used'].notna() & merged['earliest_admit'].notna(),
                                              merged['mention_time_used'] <= merged['earliest_admit'],
                                              False)
merged['mention_during_admission_window'] = np.where(merged['mention_time_used'].notna() & merged['earliest_admit'].notna() & merged['latest_discharge'].notna(),
                                                    (merged['mention_time_used'] >= merged['earliest_admit']) & (merged['mention_time_used'] <= merged['latest_discharge']),
                                                    False)
merged['mention_after_latest_discharge'] = np.where(merged['mention_time_used'].notna() & merged['latest_discharge'].notna(),
                                                   merged['mention_time_used'] > merged['latest_discharge'],
                                                   False)

# 6) Summaries to quantify potential leakage
total_subjects = len(merged)
mention_exists_count = merged['mention_exists'].sum()
no_adm_info_count = merged['no_admission_info'].sum()

# Among subjects with mentions:
m = merged[merged['mention_exists']]
m_count = len(m)
m_before_admit = m['mention_before_any_admit'].sum()
m_during = m['mention_during_admission_window'].sum()
m_after = m['mention_after_latest_discharge'].sum()
m_no_adm = m['no_admission_info'].sum()

print("\n--- Overall counts ---")
print(f"Total subjects in final_hosp: {total_subjects}")
print(f"Subjects with any mention (note-derived): {mention_exists_count} ({mention_exists_count/total_subjects*100:.2f}%)")
print(f"Subjects without admission records (no_admission_info): {no_adm_info_count} ({no_adm_info_count/total_subjects*100:.2f}%)")

print("\n--- Among subjects with mentions ---")
print(f"Total with mention: {m_count}")
print(f"Mentions before first admission: {m_before_admit} ({m_before_admit/m_count*100:.2f}%)")
print(f"Mentions during admission windows: {m_during} ({m_during/m_count*100:.2f}%)")
print(f"Mentions after latest discharge: {m_after} ({m_after/m_count*100:.2f}%)")
print(f"Mentions but no admission info: {m_no_adm} ({m_no_adm/m_count*100:.2f}%)")

# 7) Now, check leakage specifically w.r.t. your labels
# How many label=1 patients have mention; of those, how many mentions happen after discharge?
lbl = merged[merged['label'] == 1]
lbl_total = len(lbl)
lbl_with_mention = lbl['mention_exists'].sum()
lbl_with_after = lbl['mention_after_latest_discharge'].sum()

print("\n--- Among labeled positives (label==1) ---")
print(f"Total labeled positive: {lbl_total}")
print(f"Labeled positives with any mention in notes: {lbl_with_mention} ({lbl_with_mention / lbl_total * 100:.2f}%)")
print(f"Labeled positives whose mention occurs AFTER latest discharge: {lbl_with_after} ({lbl_with_after / max(1,lbl_with_mention) * 100:.2f}% of those with mention)")

# 8) Save per-subject CSV for inspection
os.makedirs(os.path.dirname(OUT_SUMMARY), exist_ok=True)
cols_to_save = [
    'subject_id','label','mention_exists','htn_first_mention_time','htn_last_mention_time','mention_time_used',
    'earliest_admit','latest_discharge','n_admissions',
    'no_admission_info','mention_before_any_admit','mention_during_admission_window','mention_after_latest_discharge'
]
merged[cols_to_save].to_csv(OUT_SUMMARY, index=False)
print(f"\nSaved per-subject timing summary to: {OUT_SUMMARY}")

# 9) Short guidance on interpretation
print("\nINTERPRETATION GUIDANCE:")
print("- If a large fraction of mentions are AFTER latest_discharge, that suggests note-derived features may be written after hospitalization and could leak outcome information.")
print("- If most mentions are DURING admission window or BEFORE earliest_admit, they are more likely predictive or represent prior history (safer).")
print("- Use the saved CSV to inspect individual examples where mention occurs after discharge (look at mention_time_used, latest_discharge, and raw notes if needed).")


Loading files...
disch rows: 86814, admissions rows: 546028, final_hosp rows: 36899
Aggregated admissions (per subject) sample:
   subject_id      earliest_admit    latest_discharge  n_admissions
0    10000032 2180-05-06 22:23:00 2180-08-07 17:50:00             4
1    10000068 2160-03-03 23:16:00 2160-03-04 06:26:00             1
2    10000084 2160-11-21 01:56:00 2160-12-28 16:07:00             2
3    10000108 2163-09-27 23:17:00 2163-09-28 09:04:00             1
4    10000117 2181-11-15 02:05:00 2183-09-21 16:30:00             2

--- Overall counts ---
Total subjects in final_hosp: 36899
Subjects with any mention (note-derived): 14392 (39.00%)
Subjects without admission records (no_admission_info): 0 (0.00%)

--- Among subjects with mentions ---
Total with mention: 14392
Mentions before first admission: 186 (1.29%)
Mentions during admission windows: 14205 (98.70%)
Mentions after latest discharge: 2 (0.01%)
Mentions but no admission info: 0 (0.00%)

--- Among labeled positives (label==